# 15 | LangChain中的短期记忆（RunnableWithMessageHistory）

前面两篇已经把 `checkpointer` 这条线讲完了：`InMemorySaver` 适合本地演示，`SqliteSaver` 和 `PostgresSaver` 适合把 Agent 的状态真正存下来。

但如果我们现在只是写了一个普通的 LCEL chain，比如：

```text
prompt | llm | parser
```

这时候为了让它“记住上一轮聊了什么”，直接上 LangGraph 的 checkpointer 就有点重了。

`RunnableWithMessageHistory` 解决的就是这个问题：**不改原来的 chain，只在外面包一层，让它按会话保存聊天记录。**

## 一、先把位置摆清楚

它和前两篇不是同一个层级的东西。

前两篇的 `InMemorySaver`、`SqliteSaver`、`PostgresSaver` 保存的是 Agent 的 `AgentState`。里面不只有消息，还可能有工具调用、中间结果、分支状态、checkpoint 等。

`RunnableWithMessageHistory` 要轻很多。它主要关心一件事：这一轮回答时，要不要把前几轮 human / ai 的消息一起带进 Prompt。

可以这么选：

- 只是普通聊天链、RAG 问答链、轻量客服助手：用 `RunnableWithMessageHistory`
- 已经在用 `create_agent(...)` 或 LangGraph：用 `checkpointer`
- 需要保存工具调用过程、恢复执行、时间回缩：还是用 `checkpointer`

所以这篇不是推翻前两篇，而是补上另一种更轻的用法。

## 二、场景：售后对话里的“这个”

看一个很常见的售后对话。

第一轮用户说：

```text
我昨天买的蓝牙耳机还没发货，想退掉。
```

第二轮用户可能只问：

```text
那这个多久能退回来？
```

人当然知道“这个”指的是刚才那笔未发货的蓝牙耳机退款。但普通 chain 如果没有历史消息，就只看到第二句话，很容易答得很飘。

`RunnableWithMessageHistory` 的用处就在这里：让这条普通 chain 在第二轮调用时，自动带上第一轮的对话。它不炫技，但很实用。

## 三、先写一个普通 chain

先不要管记忆，写一个最普通的客服 chain。

这里唯一要提前留好的位置是 `MessagesPlaceholder("chat_history")`。后面历史消息就会被塞到这里。

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 读取 .env 里的模型配置。
load_dotenv()

# 初始化聊天模型。这里继续沿用前面笔记里的百炼 OpenAI 兼容接口。
llm = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)

# chat_history 是历史消息的插入口。
# 变量名可以自己取，但后面 history_messages_key 要和它保持一致。
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个电商售后助手。回答要简洁、准确，先安抚用户，再说明下一步。",
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

# 到这里为止，它仍然只是普通 LCEL chain，还不会自动记历史。
base_chain = prompt | llm | StrOutputParser()

## 四、包上 `RunnableWithMessageHistory`

这一节是重点。

先别急着看参数，可以先把它想成一个“夹在用户调用和原始 chain 中间的小管家”。每次你调用它，它会做几件事：

1. 先从 `config` 里拿到 `session_id`
2. 用这个 `session_id` 找到对应的聊天记录
3. 把聊天记录放进 Prompt 的 `chat_history`
4. 等模型回答完，再把本轮 human 输入和 ai 回复写回记录里

也就是说，历史不是模型自己记住的，是这个包装器每次帮你读出来、塞进去、再写回去。

下面先用内存字典演示。生产里当然可以换 Redis、Postgres 之类的存储，但学习这个概念时，内存版最清楚。

In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 用字典模拟“会话仓库”。
# 一个 session_id 对应一本聊天记录。
# 注意：这是内存里的，程序一重启就没了。
session_store: dict[str, BaseChatMessageHistory] = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """拿到某个 session_id 对应的聊天记录。"""
    # 第一次见到这个 session_id，就给它新建一本空记录。
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()

    # 之后同一个 session_id 再来，就会拿到同一本记录。
    return session_store[session_id]


conversation_chain = RunnableWithMessageHistory(
    # 原始 chain：真正负责生成回答。
    base_chain,

    # 历史记录从哪里来：根据 session_id 去取。
    get_session_history,

    # 用户本轮输入字段。后面 invoke 时会传 {"input": "..."}。
    input_messages_key="input",

    # 历史消息要放进 Prompt 的哪个占位符。
    # 这里要和 MessagesPlaceholder(variable_name="chat_history") 对上。
    history_messages_key="chat_history",
)

这里最容易搞混的是 `InMemoryChatMessageHistory` 和 `RunnableWithMessageHistory` 的分工。

`InMemoryChatMessageHistory()` 返回的是一份内存里的消息记录，里面有 `messages`。它像一本记录本，本身不会主动跑去记录对话。

真正负责自动读写的是 `RunnableWithMessageHistory`：

```text
RunnableWithMessageHistory 每次运行
-> 根据 session_id 调用 get_session_history
-> 拿到对应的 InMemoryChatMessageHistory
-> 读取旧 messages，放进 chat_history
-> 模型回复后，把本轮 human / ai 消息追加回 messages
```

所以可以这样记：

```text
session_id：找哪一本记录本
InMemoryChatMessageHistory：记录本本身
RunnableWithMessageHistory：每次帮你翻记录、写记录的人
```

两个 key 也顺手记一下：

- `input_messages_key="input"`：本轮用户输入在哪个字段里
- `history_messages_key="chat_history"`：历史消息要塞进 Prompt 的哪个占位符

## 五、同一个 `session_id` 会接着聊

下面连续调用两轮。第二轮用户故意不说“蓝牙耳机”，看看它能不能接住上一轮。

In [ ]:
# 同一个 session_id 表示同一段会话。
# 真实业务里，它可以来自用户 id、浏览器会话 id 或工单 id。
config = {"configurable": {"session_id": "after-sale-user-001"}}

# 第一轮：用户给出完整背景。
first_reply = conversation_chain.invoke(
    {"input": "我昨天买的蓝牙耳机还没发货，想退掉。"},
    config=config,
)

print(first_reply)

In [ ]:
# 第二轮：用户没有重复商品和退款背景。
# 因为 session_id 没变，历史消息会被自动带进 Prompt。
second_reply = conversation_chain.invoke(
    {"input": "我忘记了刚刚跟你说过我想退掉什么东西了？以及那这个多久能退回来？"},
    config=config,
)

print(second_reply)

如果第二轮能答出“蓝牙耳机退款”相关内容，就说明历史已经带进去了。

也可以直接把记录本翻出来看看。

In [ ]:
# 看一下这个 session_id 下实际存了哪些消息。
history = get_session_history("after-sale-user-001")

for message in history.messages:
    # message.type 一般是 human 或 ai。
    print(message.type, ":", message.content[:120])

## 六、换一个 `session_id` 就是新会话

如果换成另一个 `session_id`，就相当于另一个用户打开了新的聊天窗口。

它不应该知道上一个用户聊过什么。客服系统要是串台，那就不是智能，是事故。

In [ ]:
# 新 session_id，不会读取 after-sale-user-001 的历史。
new_user_config = {"configurable": {"session_id": "after-sale-user-002"}}

new_user_reply = conversation_chain.invoke(
    {"input": "我刚刚有跟你说过我想退什么我买过的东西吗？"},
    config=new_user_config,
)

print(new_user_reply)

这个新会话没有前文背景，所以模型大概率会要求用户补充商品或订单信息。

这就对了：同一个会话要连贯，不同会话要隔离。

## 七、落地时怎么想

`RunnableWithMessageHistory` 不替你决定历史存哪里。它只要求你能通过 `session_id` 拿到一份 `BaseChatMessageHistory`。

学习和 Demo 阶段，用 `InMemoryChatMessageHistory` 最直观；简单 Web 聊天可以把历史放到 Redis 或数据库；如果已经是企业级 Agent，要保存工具调用、中间状态、恢复执行，那就回到前两篇的 `checkpointer`。

最后我会这么记：

```text
普通 chain 需要上下文连续性：RunnableWithMessageHistory
Agent 需要完整状态管理：checkpointer
```